# Set up 

In [1]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
df = pd.read_csv("../data/clean/spotify_features.csv")

In [3]:
df.head()

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,...,album_total_tracks,album_type,track_duration_min,release_year,primary_genre,genre_grouped,relative_track_position,release_decade,log_artist_followers,high_popularity
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,False,Britney Spears,80.0,17755451.0,pop,325wcm5wMnlfjmKZ8PXIIn,...,58,compilation,3.55,2009,pop,pop,0.982759,2000s,16.692203,0
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,False,BUNT.,69.0,293734.0,stutter house,2ArRQNLxf9t0O0gvmG5Vsj,...,1,single,2.65,2023,stutter house,Other,1.000000,2020s,12.590433,0
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,False,Taylor Swift,100.0,145396321.0,Unknown,4hDok0OAJd57SGIT8xuWJH,...,26,album,3.76,2021,Unknown,Unknown,0.423077,2020s,18.794974,0
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,True,Billie Eilish,90.0,118692183.0,Unknown,0JGOiO34nwfUdDrD612dOp,...,16,album,2.64,2021,Unknown,Unknown,0.125000,2020s,18.592044,1
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,False,Rihanna,90.0,68997177.0,Unknown,5QG3tjE5L9F6O2vCAPph38,...,13,album,4.45,2010,Unknown,Unknown,0.538462,2010s,18.049576,0


----

# Creating 'track_group' column
### Label that says: "this lines are the exact same music"

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8775 entries, 0 to 8774
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   track_id                 8775 non-null   str    
 1   track_name               8775 non-null   str    
 2   track_number             8775 non-null   int64  
 3   track_popularity         8775 non-null   int64  
 4   explicit                 8775 non-null   bool   
 5   artist_name              8773 non-null   str    
 6   artist_popularity        8772 non-null   float64
 7   artist_followers         8772 non-null   float64
 8   artist_genres            8775 non-null   str    
 9   album_id                 8775 non-null   str    
 10  album_name               8775 non-null   str    
 11  album_release_date       8775 non-null   str    
 12  album_total_tracks       8775 non-null   int64  
 13  album_type               8775 non-null   str    
 14  track_duration_min       8775 non-n

In [5]:
df.isna().sum()

track_id                   0
track_name                 0
track_number               0
track_popularity           0
explicit                   0
artist_name                2
artist_popularity          3
artist_followers           3
artist_genres              0
album_id                   0
album_name                 0
album_release_date         0
album_total_tracks         0
album_type                 0
track_duration_min         0
release_year               0
primary_genre              0
genre_grouped              0
relative_track_position    0
release_decade             0
log_artist_followers       3
high_popularity            0
dtype: int64

In [6]:
df['artist_name'] = df['artist_name'].fillna('Unknown')   

In [7]:
df['track_group'] = df['track_name'] + " " + df['artist_name']

In [8]:
df["track_group"].nunique()

7924

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8775 entries, 0 to 8774
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   track_id                 8775 non-null   str    
 1   track_name               8775 non-null   str    
 2   track_number             8775 non-null   int64  
 3   track_popularity         8775 non-null   int64  
 4   explicit                 8775 non-null   bool   
 5   artist_name              8775 non-null   str    
 6   artist_popularity        8772 non-null   float64
 7   artist_followers         8772 non-null   float64
 8   artist_genres            8775 non-null   str    
 9   album_id                 8775 non-null   str    
 10  album_name               8775 non-null   str    
 11  album_release_date       8775 non-null   str    
 12  album_total_tracks       8775 non-null   int64  
 13  album_type               8775 non-null   str    
 14  track_duration_min       8775 non-n

-----

## Train Test Split

### Feature set: Model A

Model A uses only characteristics of the track and its release, as defined in '05_feature_engineering.ipynb'. 
It excludes 'artist_popularity' and 'artist_followers', following the main research question in '00_business_background.ipynb': whether track popularity can be predicted without relying on the artist's existing fame.

Model B, which adds the two artist-level features, will be built afterwards as a benchmark.

**Target:** 'high_popularity' (classification).

In [10]:
# X-y split; features = X, target = y
model_a_features = ["track_duration_min", "album_total_tracks", "relative_track_position", "explicit", "genre_grouped", "album_type", "release_decade"]
features = df[model_a_features]

target = df["high_popularity"]    #what we want to predict  

In [11]:
features.columns.tolist()

['track_duration_min',
 'album_total_tracks',
 'relative_track_position',
 'explicit',
 'genre_grouped',
 'album_type',
 'release_decade']

### Grouped split

The dataset contains 697 songs that appear under more than one 'track_id': typically the album version and the single release. 
A random split would place copies of the same song in both the training and test sets, letting the model recognise a track it has already seen rather than predict it.

'GroupShuffleSplit' is used instead, grouping by 'track_group' so that all versions of a song fall entirely on one side of the split.

In [12]:
from sklearn.model_selection import GroupShuffleSplit

In [13]:
groups = df["track_group"]

In [14]:
splitter = GroupShuffleSplit(test_size=0.20, random_state=0)

In [15]:
# now we have to know where to "cut": 
## GroupShuffleSplit returns row positions, not the data itself
## we use .iloc to retrieve the corresponding rows

In [16]:
train_idx, test_idx = next(splitter.split(features, target, groups=groups))

In [17]:
print(train_idx[:10])

[ 0  2  4  8  9 10 11 12 13 14]


In [18]:
X_train = features.iloc[train_idx]   # feature rows used to train the model
X_test  = features.iloc[test_idx]    # feature rows for evaluation
y_train = target.iloc[train_idx]     # the correct answers for the training rows
y_test  = target.iloc[test_idx]      # the correct answers for the held-back rows

In [19]:
X_train.head(10)

,track_duration_min,album_total_tracks,relative_track_position,explicit,genre_grouped,album_type,release_decade
0,3.55,58,0.982759,False,pop,compilation,2000s
2,3.76,26,0.423077,False,Unknown,album,2020s
4,4.45,13,0.538462,False,Unknown,album,2010s
8,3.81,16,0.937500,False,soft pop,album,2010s
9,2.58,1,1.000000,False,Unknown,single,2020s
10,3.68,19,0.789474,False,pop,album,2010s
11,1.62,1,1.000000,True,Other,single,2020s
12,3.47,13,0.846154,False,pop,album,2000s
13,3.96,1,1.000000,False,edm,single,2020s
14,3.13,1,1.000000,False,Unknown,single,2020s


In [20]:
X_train.shape

(6996, 7)

In [21]:
X_test.shape

(1779, 7)

### The resulting split is 6,996 training rows and 1,779 test rows, which is ~79.7% / ~20.3% rather than exactly 80/20. 
> This is expected: 'GroupShuffleSplit' divides the data by groups rather than by rows, and groups vary in size, so the proportion is approximate.

#### The cell below confirms that no song appears on both sides of the split:

In [22]:
train_songs = groups.iloc[train_idx]
test_songs  = groups.iloc[test_idx]

print("Songs in both sets:", test_songs.isin(train_songs).sum())

Songs in both sets: 0


### Encoding the categorical features

'OneHotEncoder' turns each categorical column into binary indicator columns, one per category. 'drop='first' removes one category from each feature to avoid the dummy variable trap, where the dropped category is perfectly predictable from the remaining ones.

The encoder is fitted on the training set only. Fitting it on the full dataset would let information about the test set into the training pipeline. '.transform()' is then applied to both sets, so the test set is described using categories the encoder learned from the training data.

Four categorical features become 42 columns.

In [23]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first') # To avoid having an sparse_matrix as output

model_a_categorical = ['explicit', 'genre_grouped', 'album_type', 'release_decade']

ohe.fit(X_train[model_a_categorical]) # The .fit() method determines the unique values of each column
X_train_trans_np = ohe.transform(X_train[model_a_categorical])
X_train_trans_np

array([[0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 1., 0.],
       ...,
       [1., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       [1., 0., 0., ..., 0., 1., 0.]], shape=(6996, 42))

In [24]:
X_train_trans_df = pd.DataFrame(X_train_trans_np, columns=ohe.get_feature_names_out(), index=X_train.index)
X_train_trans_df

,explicit_True,genre_grouped_Unknown,genre_grouped_anime,genre_grouped_art pop,genre_grouped_art rock,genre_grouped_bedroom pop,genre_grouped_celtic,genre_grouped_classic rock,genre_grouped_country,genre_grouped_dark r&b,...,genre_grouped_soundtrack,album_type_compilation,album_type_single,release_decade_1960s,release_decade_1970s,release_decade_1980s,release_decade_1990s,release_decade_2000s,release_decade_2010s,release_decade_2020s
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8770,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8771,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8772,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8773,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [25]:
X_test_trans_np = ohe.transform(X_test[model_a_categorical])
X_test_trans_np

array([[0., 0., 0., ..., 0., 0., 1.],
       [1., 1., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [1., 1., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 1., 0.]], shape=(1779, 42))

In [26]:
X_test_trans_df = pd.DataFrame(X_test_trans_np, columns=ohe.get_feature_names_out(), index=X_test.index)
X_test_trans_df

,explicit_True,genre_grouped_Unknown,genre_grouped_anime,genre_grouped_art pop,genre_grouped_art rock,genre_grouped_bedroom pop,genre_grouped_celtic,genre_grouped_classic rock,genre_grouped_country,genre_grouped_dark r&b,...,genre_grouped_soundtrack,album_type_compilation,album_type_single,release_decade_1960s,release_decade_1970s,release_decade_1980s,release_decade_1990s,release_decade_2000s,release_decade_2010s,release_decade_2020s
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8763,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8765,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
8766,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8767,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### Scaling the numeric features

'StandardScaler' centres each numeric feature at zero with unit variance. 'album_total_tracks' contains outliers — compilations with very high track counts — and 'StandardScaler' handles those better than 'MinMaxScaler', which would squeeze the rest of the distribution into a narrow band at the bottom of the range.

As with the encoder, the scaler is fitted on the training set only.

The one-hot columns are already 0/1 by construction, so they are not scaled.

In [27]:
scaler = StandardScaler()
model_a_numerical = ['track_duration_min', 'album_total_tracks', 'relative_track_position']

scaler.fit(X_train[model_a_numerical])

X_train_scaled_np = scaler.transform(X_train[model_a_numerical])
X_test_scaled_np  = scaler.transform(X_test[model_a_numerical])

X_train_standarized = pd.DataFrame(X_train_scaled_np, columns=model_a_numerical, index=X_train.index)
X_test_standarized  = pd.DataFrame(X_test_scaled_np, columns=model_a_numerical, index=X_test.index)

In [28]:
X_train_standarized.describe()

,track_duration_min,album_total_tracks,relative_track_position
count,6.996000e+03,6.996000e+03,6.996000e+03
mean,4.895392e-16,1.955110e-17,-2.092221e-16
std,1.000071e+00,1.000071e+00,1.000071e+00
min,-3.158409e+00,-1.079826e+00,-1.582916e+00
25%,-5.749788e-01,-6.614865e-01,-9.240298e-01
50%,-4.420125e-02,-7.581065e-02,-1.364741e-01
75%,4.583934e-01,2.588613e-01,1.138881e+00
max,9.401760e+00,1.398041e+01,1.326129e+00


In [29]:
X_train_full = pd.concat([X_train_trans_df, X_train_standarized], axis=1)
X_test_full  = pd.concat([X_test_trans_df, X_test_standarized], axis=1)

In [30]:
X_train_full.shape

(6996, 45)

In [31]:
X_test_full.shape

(1779, 45)

-----


## Modelling

The preprocessing above produces 45 features: 42 one-hot columns and 3 standardised numeric columns.

In [32]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_full, y_train)
print("Dummy accuracy:", dummy.score(X_test_full, y_test))

Dummy accuracy: 0.7346824058459809


In [33]:
# univariate linear association between the numeric features and the raw popularity score
train_rows = df.iloc[train_idx]
train_rows[model_a_numerical + ["track_popularity"]].corr()["track_popularity"].round(3)

track_duration_min         0.108
album_total_tracks        -0.052
relative_track_position   -0.095
track_popularity           1.000
Name: track_popularity, dtype: float64

### Model 1 — Random Forest

Two reasons for starting with a tree-based ensemble rather than a linear model.

1.**The relationships in this dataset are not linear.** The Pearson correlations between the numeric features and track_popularity on the training set are close to zero (see the cell above). Pearson correlation only measures linear association, so a weak coefficient does not rule out a relationship, it rules out a straight-line one. A decision tree splits each variable into intervals and can capture patterns a linear model cannot represent.

2.**It answers MLQ3 directly.** Random forests expose 'feature_importances_', which is the ranking the third sub-question asks for.

The features were scaled in the previous step. Tree-based models do not need scaling, since a split point is unaffected by a linear rescaling, but scaling does them no harm and keeps the same feature matrix usable for a logistic regression comparison later.

In [34]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=0)
rf.fit(X_train_full, y_train)
print("RF accuracy:", rf.score(X_test_full, y_test))

RF accuracy: 0.6942102304665543


### Evaluation

On an imbalanced target, accuracy is a misleading single metric: a classifier can score highly while barely identifying the minority class. The confusion matrix and the per-class metrics are what the research question actually needs.

The row for class 1 is the one that matters. 

**Recall** answers: of the tracks that really are hits, how many did the model find? 

**Precision** answers: when the model predicts a hit, how often is it right?

In [35]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf.predict(X_test_full)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1109  198]
 [ 346  126]]
              precision    recall  f1-score   support

           0       0.76      0.85      0.80      1307
           1       0.39      0.27      0.32       472

    accuracy                           0.69      1779
   macro avg       0.58      0.56      0.56      1779
weighted avg       0.66      0.69      0.67      1779



In [36]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("Tracks predicted as hits:", fp + tp)
print("Of those, real hits:", tp, "->", round(tp / (fp + tp), 3))
print("Base rate of hits in the test set:", round(y_test.mean(), 3))
print("Lift:", round((tp / (fp + tp)) / y_test.mean(), 2))

Tracks predicted as hits: 324
Of those, real hits: 126 -> 0.389
Base rate of hits in the test set: 0.265
Lift: 1.47


### Results — Model A

The dummy classifier got 0.735 accuracy. The Random Forest got 0.694, so it did worse.

That looks bad but it is not a defect of the model. The dummy never predicts a hit, so it never gets a hit wrong. The Random Forest predicted 324 hits and was wrong on 198 of them, and every one of those mistakes costs accuracy.

Accuracy is not the right thing to look at here. Hits are 26.5% of the test set (472 out of 1779). Of the 324 tracks the model called hits, 126 really were hits, which is 38.9%. So when the model says "hit", it is right about 1.5 times more often than picking a track at random. The dummy gives us no way of finding a hit at all, because it never predicts one.

Still, the model only found 126 of the 472 real hits. That is a recall of 0.27.

> The intrinsic track features do carry some signal, but it is weak. They are not enough to predict hits reliably. This answers the main research question: **track popularity cannot be predicted from a track's own characteristics alone with useful accuracy.**

Model B adds the two artist-level features and will show how much of the gap they close.

----


## Model A — what we have so far

Model A uses only intrinsic track features. It gets 0.694 accuracy, below the 0.735 baseline, and finds 27% of the real hits.

> So the answer to the main research question is **"no"**: **a track's own characteristics are not enough to predict whether it becomes popular. There is some signal there, but it is weak.**

Some limitations: We only tested one model, with the default parameters, and on a single train/test split. Most tracks have no genre, so those columns add less than their number suggests. And the top 25% threshold was our decision, not something the data imposed.

But the **main limitation is the target**. 'track_popularity' measures how much a track is streamed now, not how well it did when it came out, so it is not really a measure of whether a release succeeded.

#### Next step: Model B, with the artist features added, to see if it does any better.